In [1]:
import numpy as np

m = 0.15

In [2]:
with open('hollins.dat', 'r') as f:
    lines = f.readlines()

In [3]:
urls = {}
for i in range(1, 6013):
    parts = lines[i].strip().split(' ')
    page_id = int(parts[0])
    url = parts[1]
    if page_id <= 300:
        urls[page_id] = url

print(f"Loaded {len(urls)} URLs")

# urls = {
#     1: 'http://www1.hollins.edu/',
#     2: 'http://www.hollins.edu/',
#     3: 'http://www1.hollins.edu/Docs/CompTech/Network/webmail_faq.htm',
#     ...
# }

Loaded 300 URLs


In [4]:
links = []
for i in range(6013, len(lines)):
    parts = lines[i].strip().split()
    source = int(parts[0])
    target = int(parts[1])
    
    # We Only keep links between pages 1-300
    if source <= 300 and target <= 300:
        links.append((source, target))

print(f"Total links between first 300 pages: {len(links)}")

Total links between first 300 pages: 3691


In [5]:
outgoing_count = {}
for source, target in links:
    if source not in outgoing_count:
        outgoing_count[source] = 0
    outgoing_count[source] += 1

print(f"Pages with outgoing links: {len(outgoing_count)}")

Pages with outgoing links: 253


In [6]:
# Build Matrix A
A = np.zeros((300, 300))

for source, target in links:
    A[target - 1, source - 1] = 1 / outgoing_count[source]

In [7]:
col_sums = A.sum(axis=0)
dangling = np.where(col_sums == 0)[0]
print(f"Dangling nodes: {len(dangling)}")

Dangling nodes: 47


In [10]:
for j in dangling:
    A[:, j] = 1 / 300

print(f"Fixed Dangling nodes: {len(dangling)}")

Fixed Dangling nodes: 47


In [13]:
# Build Matrix M

S = np.ones((300, 300)) / 300
M = (1 - m) * A + m * S

In [16]:
x = np.ones(300) / 300  # 1 / 300

for i in range(30):
    x = M @ x

print(f"Eigenvector x (sum = {x.sum():.4f})")

Eigenvector x (sum = 1.0000)


In [19]:
top_indices = np.argsort(x)[::-1][:10]

print("Top 10 Pages:")
for i, idx in enumerate(top_indices):
    print(f"{i+1}. Page {idx+1}: {x[idx]:.4f}")

Top 10 Pages:
1. Page 2: 0.0512
2. Page 37: 0.0405
3. Page 38: 0.0377
4. Page 61: 0.0369
5. Page 52: 0.0354
6. Page 43: 0.0307
7. Page 27: 0.0267
8. Page 28: 0.0240
9. Page 29: 0.0183
10. Page 73: 0.0155
